# 03 · Results

Charts of the out-of-sample backtest. Everything comes from `results/<stage>/test_orders.csv`,
written once by `scripts/backtest.py`; the tables are the same ones `scripts/report.py`
writes to `summary.md`. Δ vs VWAP is each strategy's implementation shortfall minus VWAP's
on the same parent order: **below zero = cheaper than VWAP**. Intervals are 95% block-bootstrap
intervals.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from adaptive_exec.pipeline import load_config
from adaptive_exec.plotting import (AXIS, INK_2, SERIES, SURFACE, DIVERGING, apply_style,
                                    ci_dots, small_multiples, zero_line)

CONFIG = os.environ.get("ADAPTIVE_EXEC_CONFIG", "configs/stage1_itch.yaml")
cfg = load_config(CONFIG)
apply_style()
pd.set_option("display.width", 160, "display.max_columns", 30)
print(f"config: {CONFIG} | stage: {cfg['stage']} | symbols: {', '.join(cfg['symbols'])}")

In [ ]:
from adaptive_exec.reporting import (ADAPTIVE, COST_PARTS, FAMILIES, MAIN_KINDS, ci,
                                    decomposition_table, load_test_orders, main_table, primary,
                                    success)

assert (Path(cfg["results_dir"]) / "test_orders.csv").exists(), "run scripts/backtest.py first"
d_all = load_test_orders(cfg)
d_main = d_all[d_all["kind"].isin(MAIN_KINDS)]
prim = primary(d_main, cfg)
t = cfg["tuning"]
display(Markdown(f"**Main table, {t['impact']} impact, {t['fill_mode']} fills**\n\n" + main_table(prim, cfg)))
display(Markdown(success(d_main, cfg)[0]))

## Δ vs VWAP by strategy

In [ ]:
names = [s for s in FAMILIES if s != "VWAP"]
est = pd.DataFrame([ci(prim[prim["strategy"] == s], cfg) for s in names], index=names,
                   columns=["delta_bps", "lo", "hi"])
fig, ax = plt.subplots(figsize=(6.4, 2.8), layout="constrained")
ci_dots(ax, est.index, est["delta_bps"], est["lo"], est["hi"])
ax.set_xlabel("Δ implementation shortfall vs VWAP (bps); below zero = cheaper")
ax.set_title(f"Paired difference vs VWAP, {t['impact']} impact, {t['fill_mode']} fills")
plt.show()
est.round(3)

## Where the cost comes from

Mean cost per component, in bps of arrival notional. Resting fills earn the spread (negative) but can pay in drift (adverse selection).

In [ ]:
g = prim.groupby("strategy")[COST_PARTS + ["is_bps"]].mean().reindex(FAMILIES)
fig, axes = small_multiples(len(COST_PARTS), ncols=5, sharex=False, sharey=True, panel=(2.3, 2.6))
for ax, part in zip(axes, COST_PARTS):
    ax.barh(g.index[::-1], g[part].to_numpy()[::-1], height=0.55, color=SERIES[0])
    zero_line(ax, "x")
    ax.grid(axis="y", visible=False)
    ax.set_title(part.replace("_bps", ""))
fig.suptitle("Mean cost by component (bps)", x=0.01, ha="left", fontweight="semibold")
plt.show()
g.round(3)

## Robustness: impact setting and fill assumption

In [ ]:
ml = ["ML adaptive (tactic only)", "ML adaptive (tactic + speed)"]
fig, axes = small_multiples(len(ml), ncols=2, sharex=True, sharey=True, panel=(4.0, 2.8))
rows = []
for ax, s in zip(axes, ml):
    labels, est = [], []
    for fill in cfg["fill_modes"]:
        for imp in cfg["impact"]:
            x = d_main[(d_main["strategy"] == s) & (d_main["impact"] == imp) & (d_main["fill_mode"] == fill)]
            labels.append(f"{imp}, {fill}")
            est.append(ci(x, cfg))
            rows.append({"strategy": s, "impact": imp, "fills": fill, **dict(zip(["delta_bps", "lo", "hi"], est[-1]))})
    e = np.array(est)
    ci_dots(ax, labels, e[:, 0], e[:, 1], e[:, 2])
    ax.set_title(s)
    ax.set_xlabel("Δ vs VWAP (bps)")
plt.show()
pd.DataFrame(rows).round(3)

## Leave-one-stock-out (Stage 1 only)

The same test orders, with the model and thresholds chosen on the *other* stocks versus on the same stock.

In [ ]:
loso = primary(d_all[d_all["kind"] == "loso"], cfg)
if loso.empty:
    print("no leave-one-stock-out units in this config")
else:
    own = prim[prim["kind"] == "per_stock"]
    syms = sorted(loso["symbol"].unique())
    fig, axes = small_multiples(len(ml), ncols=2, sharex=True, sharey=True, panel=(4.0, 2.6))
    rows = []
    for ax, s in zip(axes, ml):
        a = [ci(loso[(loso["strategy"] == s) & (loso["symbol"] == y)], cfg)[0] for y in syms]
        b = [ci(own[(own["strategy"] == s) & (own["symbol"] == y)], cfg)[0] for y in syms]
        yy = np.arange(len(syms))[::-1]
        ax.hlines(yy, np.minimum(a, b), np.maximum(a, b), color=AXIS, linewidth=2, zorder=1)
        ax.plot(b, yy, "o", color=SERIES[0], label="trained on the same stock", zorder=3)
        ax.plot(a, yy, "o", color=SERIES[1], label="trained on the other stocks", zorder=3)
        ax.set_yticks(yy, syms)
        ax.grid(axis="y", visible=False)
        zero_line(ax, "x")
        ax.set_title(s)
        ax.set_xlabel("Δ vs VWAP (bps)")
        rows += [{"strategy": s, "stock": y, "same_stock": bb, "other_stocks": aa} for y, aa, bb in zip(syms, a, b)]
    axes[0].legend(loc="lower left", fontsize=8.5)
    plt.show()
    display(pd.DataFrame(rows).round(3))